In [ ]:
import sys
import os
import glob
from pathlib import Path

sys.path.append(os.path.abspath(".."))

from config.electrodes import *
from config.subjects import *

from src.kmeans import *
from src.analysis import *
from src.plotting import *

# --------------------
import scipy.stats
import numpy as np
rng = np.random.default_rng()
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# --------------------
import mne

# --------------------
from typing import List
import pickle

In [ ]:
out_dir_tfr = Path.cwd().parent / "tfr_results"
out_dir_tfr.mkdir(exist_ok=True)

In [ ]:
### store files in a list ###
data_path = "../data/data_epochs_longer"
data_files = glob.glob(f"{data_path}/*-epo.fif")
data_files.sort()  # keep an ascending order
print(data_files)

In [ ]:
# ------- Extract metadata info once (necessary for functions) ----------#
sample_epoch = mne.read_epochs(data_files[2], preload=False, verbose=0)
info = sample_epoch.info
times = sample_epoch.times

### electrodes of interest ###
picks_sp, picks_erp, picks_all = set_electrodes(elecs_interest_sp, elecs_interest_erp, info)

### psd and freq info ###
psd_tmp = psd(sample_epoch, fmin=1, fmax=40, picks=picks_sp)
freqs = psd_tmp.freqs
log_freqs = np.log10(freqs)
mask_spindle = (freqs >= 10) & (freqs <=16)
mask_no_spindle = (freqs < 10) | (freqs >16)
mask_tfr = (freqs >= 5) & (freqs <=20)
spindle_start, spindle_end = freqs[mask_spindle][0], freqs[mask_spindle][-1]
freqs_tfr = freqs[mask_tfr]
n_cycles = freqs_tfr / 2.0 

del sample_epoch, psd_tmp
#------------------------------------------------------------------------#

In [ ]:
log_freqs = np.log10(freqs)
mask_spindle = (freqs >= 10) & (freqs <=16)
mask_no_spindle = (freqs < 10) | (freqs >16)
mask_tfr = (freqs >= 5) & (freqs <=20)
spindle_start, spindle_end = freqs[mask_spindle][0], freqs[mask_spindle][-1]
erp_peak_1 = 0.4
erp_peak_2 = 0.6
erp_peak_idx_1 = np.abs(times - erp_peak_1).argmin()
erp_peak_idx_2 = np.abs(times - erp_peak_2).argmin()

In [ ]:
if len(subjects) != len(mask):
    raise ValueError("subject_nums and mask must be the same length.")

# to store the results 
spindle_metrics = {}
results_kmeans = {}
spindle_index = {}
result_tfr  = {}

# for ERP
list_s = []
list_ns = []
list_ns_in_s = []

# --------- loop over each subject to extract information necessary for the analysis -------- #

for f, n, m in zip(data_files, subjects, mask):

    if m != 1:
        continue

    subj= f"S{n:02d}"
    print("Processing", subj)

    ### load epochs + PSD ###
    epoch = mne.read_epochs(f, preload=False, verbose=0)
    
    print(f"Total epochs = {len(epoch)}")
    
    # take only trials where sourdine(volume) > 0.2 ; S34 is missing metadata.
    valid_trials = (
    np.arange(200, len(epoch))
    if subj in ["S34", "S02"]
    else np.where(epoch.metadata["sourdine"].values >= 0.2)[0]
    )

    print(f"valid epochs after volume filtering = {len(valid_trials)}")

    epoch = epoch[valid_trials]

    psd_data = psd(epoch, fmin=1, fmax=40, picks=picks_sp)

    print("PSD done")

    ### calculate delta ###
    log_psd, fit, residual, delta, coef_mat = compute_spindle_metric(
        psd_data,
        log_freqs,
        mask_no_spindle,
        mask_spindle,
        axis=1 # change this to 0 if need to average over trials so that each channel = 1 value (topomap)
    )
    print("Delta calculated")

    ### ERP peak ###
    data_peak = epoch.get_data(picks=picks_erp)
    peak = data_peak[:, :, erp_peak_idx_1 : erp_peak_idx_2]
    peak_mean = peak.mean(axis=(1,2))

    print("ERP peak done")
    

    ### k-means ###

    data_kmeans = delta[:, np.newaxis]
    center_history, labels, cost_history = kmeans_numpy(
        data_kmeans,
        2,
        N_iters=10,
        seed=3
    )

    vals, counts = np.unique(labels, return_counts=True)
    # find label with fewer elements
    spindle_label = vals[np.argmin(counts)]
    # split indices
    spindles = np.where(labels == spindle_label)[0]
    no_spindles = np.where(labels != spindle_label)[0]

    print("k-means done")

    # -------- store the results --------- #
    
    spindle_metrics[subj] = {
    "log_psd" : log_psd,
    "fit": fit,
    "residual": residual,
    "delta": delta,
    "coef_mat": coef_mat,
    "peak" : peak_mean
    }

    results_kmeans[subj] = {
        "center_history": center_history,
        "labels": labels,
        "cost_history": cost_history
    }

    spindle_index[subj] = {
        "spindles" : spindles,
        "no_spindles" : no_spindles
    }

    print("storing results done")

    # ------------------------------------- #

    ### Add values for ns_in_spindles ###
    spindle_window = window_delta(spindle_metrics, spindle_index, 50)
    spindle_index = extract_ns(spindle_window, spindle_index, 0.1, 50)  

    print("window_delta and extract_ns done")

    ### ERP conditions ###

    list_s.append(epoch[spindles].average())

    ns_in_spindles = spindle_index[subj]["ns_in_spindles"]
    
    list_ns_in_s.append(epoch[ns_in_spindles].average())
    
    valid_no_spindles = np.setdiff1d(
                                        no_spindles,
                                        ns_in_spindles
                                    )
    
    spindle_index[subj]["valid_no_spindles"] = valid_no_spindles

    if len(valid_no_spindles) <= len(spindles):
            samples = valid_no_spindles
    else : 
            samples = rng.choice(valid_no_spindles, size=len(spindles), replace=False)

    evoked_ns = epoch[samples].average()
  
    list_ns.append(evoked_ns) 


    ### TFR ###

    spindles, ns, ns_in_s = spindle_index[subj]["spindles"], spindle_index[subj]["valid_no_spindles"], spindle_index[subj]["ns_in_spindles"]

    power_s, itc_s = epoch[spindles].compute_tfr(
    method="morlet",
    freqs=freqs_tfr,
    n_cycles=n_cycles,
    average=True,
    return_itc=True,
    decim=3,
)
    
    power_ns, itc_ns = epoch[valid_no_spindles].compute_tfr(
    method="morlet",
    freqs=freqs_tfr,
    n_cycles=n_cycles,
    average=True,
    return_itc=True,
    decim=3,
)
    
    power_ns_in_s, itc_ns_in_s = epoch[ns_in_s].compute_tfr(
    method="morlet",
    freqs=freqs_tfr,
    n_cycles=n_cycles,
    average=True,
    return_itc=True,
    decim=3,
)
    
    result_tfr[subj] = {}

    result_tfr[subj].update({
    "spindles": {
        "power": power_s,
        "itc": itc_s
    },
    "no_spindles": {
        "power": power_ns,
        "itc": itc_ns
    },
    "ns_in_spindles": {
        "power" : power_ns_in_s,
        "itc": itc_ns_in_s 
    }
})
    
    # Save the result in pickle files
    
    with open(out_dir_tfr / f"tfr_{subj}.pkl", "wb") as f:
        pickle.dump(result_tfr[subj], f)

    print(f"Done for {subj}")

    ### delete variables to free memory ###
    del epoch
    del psd_data
    del power_s
    del itc_s
    del power_ns
    del itc_ns
    del power_ns_in_s
    del itc_ns_in_s

    print("Loaded subjects:", spindle_metrics.keys())
    print("\n"*2)

In [ ]:
X, list_spindle, list_no_spindle, list_ns_in_s, evokeds = prep_permute_and_erp_optim(list_s, list_ns, list_ns_in_s, picks_erp)

In [2]:
# topo_delta_all(spindle_metrics)

In [ ]:
# plot_topo_delta(spindle_metrics, info)

In [ ]:
# plot_delta(spindle_metrics)

In [ ]:
# plot_freq_maxval_sp(spindle_metrics, spindle_index, "group", freqs, mask_spindle)

In [ ]:
# plot_kmeans(results_kmeans, spindle_metrics, 2)

In [ ]:
# plot_sp_ratio_window(spindle_window, spindle_index)

In [ ]:
# plot_fit(spindle_metrics, freqs, spindle_index, spindle_start, spindle_end)  

In [ ]:
### Permutation-based cluster test

adjacency, ch_names = mne.channels.find_ch_adjacency(info, "eeg")
alpha_cluster_forming = 0.001

n_conditions = len(X)
n_observations = len(X[0])
dfn = n_conditions - 1
dfd = n_observations - n_conditions

f_thresh = scipy.stats.f.ppf(1 - alpha_cluster_forming, dfn=dfn, dfd=dfd)

F_obs, clusters, cluster_p_values, H0 = mne.stats.spatio_temporal_cluster_test(
        X,
        threshold = f_thresh,
        n_permutations=1024,
        tail=1,
        n_jobs=None,
        buffer_size=None,
        adjacency=adjacency,
        seed = 10,
    )

In [ ]:
### set the p-value and identify clusters that are above this p-value

p_accept = 0.01
good_cluster_inds = np.where(cluster_p_values < p_accept)[0]

for i_clu, clu_idx in enumerate(good_cluster_inds):
    # unpack cluster information, get unique indices
    time_inds, space_inds = np.squeeze(clusters[clu_idx])
    ch_inds = np.unique(space_inds)
    time_inds = np.unique(time_inds)

In [ ]:
#visualize the results

from mpl_toolkits.axes_grid1 import make_axes_locatable


colors = {"spindle": "crimson", "no_spindle": "steelblue", "no_spindle in spindles" : "green"}

list_time_inds = []
list_ch_inds = []

for idx, i_cluster in enumerate(good_cluster_inds):
    time_inds, space_inds = clusters[i_cluster]
    ch_inds = np.unique(space_inds)
    time_inds = np.unique(time_inds)

    list_time_inds.append([time_inds])
    list_ch_inds.append([ch_inds])

    f_map = F_obs[time_inds, :].mean(axis = 0)
    sig_times = times[time_inds]

    mask = np.zeros((f_map.shape[0], 1), dtype=bool)
    mask[ch_inds, :] = True

    fig, ax_topo = plt.subplots(1, 1, figsize=(10, 3), layout="constrained")

    f_evoked = mne.EvokedArray(f_map[:, np.newaxis], info, tmin=0)
    f_evoked.plot_topomap(
        times=0,
        mask=mask,
        axes=ax_topo,
        cmap="Reds",
        vlim=(np.min, np.max),
        show=False,
        colorbar=False,
        mask_params=dict(markersize=10),
    )
    image = ax_topo.images[0]

    ax_topo.set_title("")

    divider = make_axes_locatable(ax_topo)

    ax_colorbar = divider.append_axes("right", size="5%", pad=0.05)
    plt.colorbar(image, cax=ax_colorbar)
    ax_topo.set_xlabel(
        "Averaged F-map ({:0.3f} - {:0.3f} s)".format(*sig_times[[0, -1]])
    )

    ax_signals = divider.append_axes("right", size="300%", pad=1.2)

    title = f"Cluster #{idx + 1}, {len(ch_inds)} sensors"
    
    mne.viz.plot_compare_evokeds(
        evokeds,
        title=title,
        combine="mean",
        picks=ch_inds,
        axes=ax_signals,
        colors=colors,
        show=False,
        split_legend=True,
        truncate_yaxis="auto",
    )

    # plot temporal cluster extent
    ymin, ymax = ax_signals.get_ylim()
    ax_signals.fill_betweenx(
        (ymin, ymax), sig_times[0], sig_times[-1], color="orange", alpha=0.3
    )

plt.show()

In [ ]:
## post-hoc t tests

X0_roi_list = [] 
X1_roi_list = [] 
X2_roi_list = []

for i in range(len(list_ch_inds)):

    X0_roi = X[0][:, list_time_inds[i][0]][:,:, list_ch_inds[i][0]].mean(axis=(1, 2)) # spindle
    X1_roi = X[1][:, list_time_inds[i][0]][:,:, list_ch_inds[i][0]].mean(axis=(1, 2)) # ns
    X2_roi = X[2][:, list_time_inds[i][0]][:,:, list_ch_inds[i][0]].mean(axis=(1, 2)) # ns_in_s

    X0_roi_list.append(X0_roi)
    X1_roi_list.append(X1_roi) 
    X2_roi_list.append(X2_roi)

    print(scipy.stats.ttest_rel(X0_roi, X1_roi))
    print(scipy.stats.ttest_rel(X0_roi, X2_roi))
    print(scipy.stats.ttest_rel(X1_roi, X2_roi))

In [ ]:
## result t-test visualization

palette = {
    "spindles": "red",
    "no spindles": "blue",
    "no spindle in spindles": "green"
}

df = pd.DataFrame({
    "Amplitude": np.concatenate([X0_roi_list[2], 
                                 X1_roi_list[2], 
                                 X2_roi_list[2]]),
    "Condition": ["spindles"]*len(X0_roi_list[2]) + ["no spindles"]*len(X1_roi_list[2]) + ["no spindle in spindles"]*len(X2_roi_list[2])
})

plt.figure(figsize=(5,5))

ax = sns.boxplot(
    data=df,
    x="Condition",
    y="Amplitude",
    showfliers=True,
    palette=palette
)

sns.swarmplot(
    data=df,
    x="Condition",
    y="Amplitude",
    color="black",
    size=5
)

y_max = df["Amplitude"].max()
y_min = df["Amplitude"].min()
yrange = y_max - y_min

base = y_max + 0.05 * yrange
h = 0.03 * yrange
step = 0.07 * yrange  

y1 = base

ax.plot([0, 0, 1, 1],
        [y1, y1 + h, y1 + h, y1],
        color="black")

ax.text(0.5, y1 + h, "***", ha="center", va="bottom", fontsize=14)


y2 = base + step

ax.plot([1, 1, 2, 2],
        [y2, y2 + h, y2 + h, y2],
        color="black")

ax.text(1.5, y2 + h, "****", ha="center", va="bottom", fontsize=14)


y3 = base + 2 * step

ax.plot([0, 0, 2, 2],
        [y3, y3 + h, y3 + h, y3],
        color="black",
        zorder=3)

ax.text(1.0, y3 + h, "**", ha="center", va="bottom", fontsize=14)
 

plt.show()

In [ ]:
tfr_dir = Path.cwd().parent / "tfr_results"

# dictionary to store all subjects
result_tfr = {}

for file in tfr_dir.glob("*.pkl"):
    subj = file.stem.replace("tfr_", "")   # "tfr_S01.pkl" -> "S01"

    with open(file, "rb") as f:
        result_tfr[subj] = pickle.load(f)

print(result_tfr.keys())

In [ ]:
ga_s = []
ga_ns = []
ga_ns_in_s = []

for subj, data in result_tfr.items():
    s = result_tfr[subj]["spindles"]["power"].copy().pick(picks_erp)
    ns = result_tfr[subj]["no_spindles"]["power"].copy().pick(picks_erp)
    ns_in_s = result_tfr[subj]["ns_in_spindles"]["power"].copy().pick(picks_erp)


    ga_s.append(s)
    ga_ns.append(ns)
    ga_ns_in_s.append(ns_in_s)

tfr_s_ga = mne.grand_average(ga_s)
tfr_ns_ga = mne.grand_average(ga_ns)
tfr_ns_in_s_ga = mne.grand_average(ga_ns_in_s)

In [ ]:
ga_s_itc = []
ga_ns_itc = []
ga_ns_in_s_itc = []

for subj, data in result_tfr.items():
    s = result_tfr[subj]["spindles"]["itc"].copy().pick(picks_erp)
    ns = result_tfr[subj]["no_spindles"]["itc"].copy().pick(picks_erp)
    ns_in_s = result_tfr[subj]["ns_in_spindles"]["itc"].copy().pick(picks_erp)

    ga_s_itc.append(s)
    ga_ns_itc.append(ns)
    ga_ns_in_s_itc.append(ns_in_s)

tfr_s_ga_itc = mne.grand_average(ga_s_itc)
tfr_ns_ga_itc = mne.grand_average(ga_ns_itc)
tfr_ns_in_s_ga_itc = mne.grand_average(ga_ns_in_s_itc)


In [ ]:
baseline = (-0.3, -0.1)
mode = "mean"
tmin = -0.5
tmax = 1.5


fig, axes = plt.subplots(1, 3, figsize=(18, 5))

tfr_s_ga.plot(
    baseline=baseline,
    mode=mode,
    combine="mean",
    tmin=tmin,
    tmax=tmax,
    vlim=(-5*(10**-11), 5*(10**-11) ),
    axes=axes[0],
    show=False
)
axes[0].set_title("Grand average - spindle")


tfr_ns_ga.plot(
    baseline=baseline,
    mode=mode,
    combine="mean",
    tmin=tmin,
    tmax=tmax,
    vlim=(-5*(10**-11), 5*(10**-11) ),
    axes=axes[1],
    show=False
)
axes[1].set_title("Grand average - no spindle")


tfr_ns_in_s_ga.plot(
    baseline=baseline,
    mode=mode,
    combine="mean",
    tmin=tmin,
    tmax=tmax,
    vlim=(-5*(10**-11), 5*(10**-11) ),
    axes=axes[2],
    show=False
)
axes[2].set_title("Grand average - no spindles in spindles")

In [ ]:
fig, axes = plt.subplots(1,3, figsize=(18, 5))

tfr_s_ga_itc.plot(
        baseline=baseline,
        mode=mode,
        combine="mean",
        tmin=tmin,
        tmax=tmax,
        vlim = (-0.06, 0.06),
        axes=axes[0],
        show=False
    )
axes[0].set_title("Grand average - spindle(itc)")

tfr_ns_ga_itc.plot(
        baseline=baseline,
        mode=mode,
        combine="mean",
        tmin=tmin,
        tmax=tmax,
        vlim = (-0.06, 0.06),
        axes=axes[1],
        show=False
    )
axes[1].set_title("Grand average - no spindle(itc)")

tfr_ns_in_s_ga_itc.plot(
        baseline=baseline,
        mode=mode,
        combine="mean",
        tmin=tmin,
        tmax=tmax,
        vlim = (-0.06, 0.06),
        axes=axes[2],
        show=False
    )
axes[2].set_title("Grand average - no spindles in spindles (itc)")

In [ ]:
baseline = (-0.3, -0.1)
mode = "mean"
tmin = -0.5
tmax = 1.5

n_subj = len(result_tfr)

fig, axes = plt.subplots(n_subj, 3, figsize=(15, 3 * n_subj))

for i, (subj, value) in enumerate(result_tfr.items()):

    # --- spindle condition (select electrodes first)
    power_sp = value["spindles"]["power"].copy().pick(picks_erp)

    power_sp.plot(
        baseline=baseline,
        mode=mode,
        combine="mean",
        tmin=tmin,
        tmax=tmax,
        axes=axes[i, 0],
        show=False
    )
    axes[i, 0].set_title(f"{subj} - spindle")

    # --- no spindle condition (select electrodes first)
    power_ns = value["no_spindles"]["power"].copy().pick(picks_erp)

    power_ns.plot(
        baseline=baseline,
        mode=mode,
        combine="mean",
        tmin=tmin,
        tmax=tmax,
        axes=axes[i, 1],
        show=False
    )
    axes[i, 1].set_title(f"{subj} - no spindle")

    # --- ns in spindles condition (select electrodes first)
    power_ns_in_s = value["ns_in_spindles"]["power"].copy().pick(picks_erp)

    power_ns_in_s.plot(
        baseline=baseline,
        mode=mode,
        combine="mean",
        tmin=tmin,
        tmax=tmax,
        axes=axes[i, 2],
        show=False
    )
    axes[i, 2].set_title(f"{subj} - no spindle in spindles")

plt.tight_layout()
plt.show()